# Generic Compute Engine

This notebook demos the generic compute engine end-to-end. It authenticates with ** Evo OAuth** and makes
**calls to the live compute discovery endpoint**
(`GET /compute/orgs/{org_id}/tasks`).

The thesis of the POC: a **fully generic** engine that reads the task catalogue **live from
discovery**, so a new platform task needs **no SDK release**. There is **no per-task
Python code** — every topic/task and every `run(...)` signature is synthesised from the
live schema. Execution delegates to `evo.compute.JobClient`. The engine is **async-native**, so calls are awaited (Jupyter supports top-level `await`).

What you'll see:
1. Authenticate  → a `manager` that *is* an `evo.common.IContext`.
2. `async with ComputeClient(manager) as client:` — discovery + auth happen on entry (**fail-fast**).
3. The dynamic, schema-driven namespace (`await client.<topic>.<task>.run(...)`).
4. Argument validation that happens **before** any network call.
5. The **raw** live catalogue via `DiscoveryClient` — the same in-package client the engine delegates discovery to (the capability still missing from the SDK).
6. **Live breadth vs. the generated stubs** — tasks the platform advertises *right now*
   that were never stubbed (including `feature_flag` / api-preview tasks).
7. (Optional) executing a task against Evo compute platform.

> Prereqs: `pip install -e packages/evo-sdk-common packages/evo-compute` and the notebook
> extras (`evo.notebooks`, `aiohttp`). Run this notebook from the `poc-engine/` folder
> so `poc_compute_engine` and `generate_stubs` import locally.

## 1. Authenticate (real Evo OAuth)

`ServiceManagerWidget` handles the OAuth login **and** organization/hub/workspace
selection, exactly as in the SDK's own code samples. The returned `manager` implements
`evo.common.IContext` (`get_connector()` + `get_org_id()`), which is all our engine needs.

Replace `client_id` / `redirect_url` with your Evo app credentials.

In [ ]:
from evo.notebooks import ServiceManagerWidget

client_id = "<YOUR_CLIENT_ID>"  # replace with your own client id from the Evo developer portal
redirect_url = "<YOUR_REDIRECT_URL>"  # replace with your own redirect url from the Evo developer portal

# OAuth issuer (IMS). Default is https://ims.bentley.com.
base_uri = "https://qa-ims.bentley.com"

# Evo Discovery base. The SDK appends /evo/identity/v2/discovery?service=evo, so this
# resolves to https://uat-discover.test.api.seequent.com/evo/identity/v2/discovery?service=evo
discovery_url = "https://uat-discover.test.api.seequent.com"

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    base_uri=base_uri,
    discovery_url=discovery_url,
    redirect_url=redirect_url,
    cache_location="./notebook-data",
).login()

In [ ]:
# Rich HTML display for Evo widgets (org/hub/workspace pickers above).
%load_ext evo.widgets

## 2. Construct the engine — fail-fast discovery on open

`ComputeClient` is **async-native**: opening it (here via `await ComputeClient.connect(manager)`, or `async with ComputeClient(manager) as client:`) makes the authenticated discovery call eagerly, so a missing/expired/unentitled token raises a typed SDK exception *here*, before any task namespace is handed out. There is no event-loop juggling and no `nest_asyncio` — every `run(...)` is awaited on the notebook's own loop.

In [ ]:
from poc_compute_engine import ComputeClient

client = await ComputeClient.connect(manager)   # opens + runs the discovery call now (fail-fast)
client

## 3. The live, schema-driven namespace

No per-task code: topics and tasks come straight from the live discovery response, and
each `.run` signature (parameter names, types, `Literal` enums, defaults) is synthesised
from the task's JSON schema.

In [ ]:
import inspect

topics = [t for t in dir(client) if not t.startswith("_") and t not in ("connect", "aclose", "refresh")]
print("topics:", topics)

for topic in topics:
    ns = getattr(client, topic)
    print(f"  {topic}:", dir(ns))

# Show one synthesised signature.
topic = "geostatistics"
task = dir(getattr(client, topic))[0]
runner = getattr(getattr(client, topic), task).run
print(f"\nsignature of {topic}.{task}.run:\n  ", inspect.signature(runner))

## 4. Argument validation happens *before* the network call

Required parameters are enforced from the schema, so a bad call fails locally instead of
burning a round-trip.

In [ ]:
try:
    await runner()  # await drives the coroutine body, where validation runs (no network reached)
except TypeError as exc:
    print("caught locally (no network):", exc)

## 5. A specialized typed runner (the override seam)

Most tasks ride the generic engine, but a task can opt into a **hand-written, fully-typed runner** by dropping a module at `poc_compute_engine/overrides/<topic>/<task>.py`. The engine auto-discovers it by convention and routes to it transparently — same `client...run(...)` DX.

`geostatistics/kriging-gcp` has such an override ([`overrides/geostatistics/kriging_gcp.py`](./poc_compute_engine/overrides/geostatistics/kriging_gcp.py)), so `client.geostatistics.kriging_gcp` resolves to a `KrigingGcpRunner` with bespoke validation, a `mean` parameter absent from the discovery schema, and result helpers (`summary()`, `portal_url()`). The other geostatistics tasks stay generic.

In [ ]:
import inspect

kriging = client.geostatistics.kriging_gcp
declustering = client.geostatistics.declustering
print("kriging      ->", type(kriging).__name__)        # KrigingGcpRunner (specialized override)
print("declustering ->", type(declustering).__name__)   # _TaskProxy (generic)
print("kriging.run  ->", inspect.signature(kriging.run))  # note the override-only `mean` param

# Override-only validation the JSON Schema can't express — runs BEFORE any network call
# (awaiting drives the coroutine body up to the validation, no HTTP reached):
try:
    await kriging.run(source="grade", target="kriged", variogram="vario-1", kriging_type="simple")
except ValueError as exc:
    print("validation:", exc)

In [ ]:
# The override also owns the OUTPUT side: a hand-curated, fully-typed result with
# helpers the generic TaskResult can't synthesise. Real execution needs live object
# references (see the optional execute cell below), so here we hydrate the typed result
# from a representative discovery-shaped payload to show the surface it exposes.
from poc_compute_engine.overrides.geostatistics.kriging_gcp import KrigingGcpResult

sample_payload = {
    "message": "Kriging completed.",
    "target": {
        "reference": "https://hub.evo/objects/2f1c-0000-abcd",
        "name": "grade_estimate_grid",
        "description": None,
        "schema_id": "regular-3d-grid/1.2.0",
        "attribute": {
            "reference": "cell_attributes[?name=='kriged_grade']",
            "name": "kriged_grade",
        },
    },
}
result = KrigingGcpResult(sample_payload, kriging_type="ordinary")

print("message      :", result.message)
print("target name  :", result.target.name)
print("attribute    :", result.target.attribute.name)
print("summary()    :", result.summary())       # override-only helper
print("portal_url() :", result.portal_url())    # override-only helper
print("to_dataframe :", result.target.to_dataframe())

## 6. The raw live catalogue via `DiscoveryClient`

`evo.compute` has a `JobClient` for *executing* tasks but **no discovery client** — its `TasksApi` only exposes `execute_task`. Listing tasks is the missing capability. The engine delegates discovery to `poc_compute_engine.DiscoveryClient` (the symmetric twin of `JobClient`); here we use that **same** client directly to show the raw catalogue — including api-preview `feature_flag` tasks — over the authenticated context.

In [ ]:
from poc_compute_engine import DiscoveryClient

tasks = await DiscoveryClient.from_context(manager).list_tasks()
print(f"{len(tasks)} tasks advertised by the platform:\n")
for t in sorted(tasks, key=lambda s: (s["topic"], s["name"])):
    flag = f"  [api-preview: {t['feature_flag']}]" if t.get("feature_flag") else ""
    print(f"  {t['topic']}/{t['name']:<22} v{t.get('version','?')}{flag}")

## 7. Live breadth vs. the generated stubs

The `.pyi` stubs are generated **offline** from a point-in-time snapshot in `poc_compute_engine/schemas/`
(see `generate_stubs.py`). The runtime is always live, so the platform typically advertises
**more** tasks than the snapshot knows — those run fine through the engine today, they just
aren't statically typed until stubs are regenerated. This is the generic-vs-codegen trade:
**total runtime breadth, point-in-time static breadth**.

In [ ]:
from generate_stubs import _load_bundled_specs

stubbed = {(s["topic"], s["name"]) for s in _load_bundled_specs()}
live = await DiscoveryClient.from_context(manager).task_keys()

print("stubbed (typed in __init__.pyi):")
for k in sorted(stubbed):
    print("   ", "/".join(k))

print("\nadvertised live but NOT stubbed (runnable now, no SDK release needed):")
for k in sorted(live - stubbed):
    print("   ", "/".join(k))

missing_stub = stubbed - live
if missing_stub:
    print("\nstubbed but not currently advertised:", sorted(missing_stub))

## 8. (Optional) Execute a task 

Execution is wired to the existing `evo.compute.JobClient` (submit → poll → results).
Running a real task needs **real** object/attribute references (a source attribute, a
target attribute on an output object, a variogram id, ...). Building those inputs is out
of scope for this engine demo — see the SDK's full
`code-samples/.../running-kriging-compute.ipynb` for an end-to-end example that creates a
PointSet, Variogram and BlockModel first.

> Note: the engine's `_mock_resolve` is a POC stand-in for reference resolution. In a
> production engine this step would resolve bare names against the objects service before
> submitting. Fill in real references below to actually submit.

In [ ]:
# Uncomment and supply REAL references to submit a job through evo.compute.JobClient:
#
# result = await client.geostatistics.kriging_gcp.run(
#     source="<real source attribute reference>",
#     target="<real target attribute reference>",
#     variogram="<real variogram object id>",
# )
# print(result.message)
# result.target.to_dataframe()

## 9. Cleanup

Close the engine (closes the transport; the notebook's own event loop is left running).

In [ ]:
await client.aclose()